# Embedding Quality Pipeline — Real Embeddings from Parquet

This notebook loads real 1536-dimensional embeddings from a parquet file, runs the full nlpie evaluation pipeline, and visualises the results.

In [1]:
import sys
import math

import numpy as np
import pyarrow.parquet as pq

sys.path.insert(0, "..")
sys.path.insert(0, "../../python")

from nlpie import (
    EmbeddingPreprocessor,
    compute_hubness,
    effective_rank,
    similarity_to_global_mean,
    cosine_similarity_matrix,
    explain_hubness,
    plot_hubness_histogram,
    plot_similarity_distribution,
    plot_similarity_to_mean,
)
from metrics.quality import evaluate_embedding_quality

## 1. Load Embeddings from Parquet

The file contains 323 task-description embeddings (1536-d) with their task codes.

In [2]:
table = pq.read_table("../../files/task_embeddings_llm_descriptions.parquet")
task_codes = table.column("task_code").to_pylist()
embeddings = np.array(table.column("embedding").to_pylist(), dtype=np.float32)

print(f"Loaded {len(embeddings)} embeddings of shape {embeddings.shape}")
print(f"Sample task codes: {task_codes[:3]}")

Loaded 323 embeddings of shape (323, 1536)
Sample task codes: ['32-00-00-810-801-A', '32-10-00-810-801-A', '32-10-00-810-802-A']


## 2. Preprocessing

We mean-center and L2-normalise the embeddings so that cosine similarity behaves consistently.

In [3]:
preprocessor = EmbeddingPreprocessor()

emb_centered, stats = preprocessor.mean_center(embeddings)
emb_norm = preprocessor.l2_normalize_rows(emb_centered)

print(f"Mean vector length before: {np.linalg.norm(embeddings, axis=1).mean():.4f}")
print(f"Mean vector length after:  {np.linalg.norm(np.array(emb_norm), axis=1).mean():.4f}")

Mean vector length before: 1.0000
Mean vector length after:  1.0000


## 3. Intrinsic Metrics — Cosine Similarity Distribution

The pairwise cosine similarity matrix tells us how spread out the embeddings are. A mean near 0 with high std indicates good dispersion.

In [4]:
sim_matrix = cosine_similarity_matrix(emb_norm)
n = len(sim_matrix)

values = []
for i in range(n):
    for j in range(i + 1, n):
        values.append(sim_matrix[i][j])

mean_sim = sum(values) / len(values)
std_sim = math.sqrt(sum((v - mean_sim) ** 2 for v in values) / len(values))

print(f"Pairwise cosine similarity — mean={mean_sim:.4f}  std={std_sim:.4f}")
print(f"  min={min(values):.4f}  max={max(values):.4f}")

Pairwise cosine similarity — mean=-0.0029  std=0.1770
  min=-0.4042  max=0.9809


### Plot the similarity distribution

In [5]:
fig_sim = plot_similarity_distribution(values, mean_sim, std_sim)
fig_sim.show()

## 4. Geometry & Pathology Metrics

### Effective Rank
Measures the intrinsic dimensionality of the embedding space.

In [6]:
eff_rank = effective_rank(emb_norm)
print(f"Effective rank: {eff_rank:.2f} (out of {embeddings.shape[1]} dimensions)")

Effective rank: 70.87 (out of 1536 dimensions)


### Similarity to Global Mean
How similar each point is to the global centroid. Large variance here can indicate clusters.

In [7]:
sims = similarity_to_global_mean(emb_norm)
mean_sim = sum(sims) / len(sims)
print(f"Mean similarity to centroid: {mean_sim:.4f}  range=[{min(sims):.4f}, {max(sims):.4f}]")

Mean similarity to centroid: 0.0157  range=[-0.5046, 0.5411]


In [8]:
fig_sim_mean = plot_similarity_to_mean(sims)
fig_sim_mean.show()

### Hubness Analysis

Hubness measures whether some points appear disproportionately often in K-NN lists. High skewness signals a pathological embedding space.

In [9]:
k = 10
hub_counts, skewness = compute_hubness(emb_norm, k=k)
print(f"Hubness skewness (k={k}): {skewness:.4f}")
print(f"Max hubness count: {max(hub_counts)}  Mean (theoretical): {k}")

Hubness skewness (k=10): 0.2935
Max hubness count: 22  Mean (theoretical): 10


In [10]:
fig_hub = plot_hubness_histogram(hub_counts, skewness, k)
fig_hub.show()

### Hubness Explanation

The `explain_hubness` function turns the raw skewness into a human-readable diagnosis.

In [11]:
explanation = explain_hubness(hub_counts, skewness, k, top_n=5)

print(f"Severity: {explanation.severity}")
print(f"Summary: {explanation.summary}")
print(f"Interpretation: {explanation.interpretation}")
print(f"\nTop hubs:")
for hub in explanation.top_hubs:
    print(f"  Point {hub.index}: count={hub.count}  share={hub.share_of_total:.2%}")

Severity: low
Summary: Low hubness (skewness=0.29). The space is fairly well-behaved.
Interpretation: The embedding space is fairly well-behaved. Slight skew is present but unlikely to cause practical issues for most downstream tasks.

Top hubs:
  Point 204: count=22  share=0.68%
  Point 21: count=21  share=0.65%
  Point 25: count=21  share=0.65%
  Point 120: count=21  share=0.65%
  Point 199: count=21  share=0.65%


## 5. Clustering Metrics

We derive cluster labels from the hierarchical task codes (first segment). This lets us evaluate ARI, NMI, Purity, Silhouette, and Calinski-Harabasz.

In [12]:
from nlpie import (
    adjusted_rand_index,
    normalized_mutual_info,
    purity_score,
    calinski_harabasz_score,
    silhouette_score,
)

# Use the first segment of the task code as a coarse label
label_map = {}
labels = []
for code in task_codes:
    prefix = code.split("-")[0]
    if prefix not in label_map:
        label_map[prefix] = len(label_map)
    labels.append(label_map[prefix])

print(f"Derived {len(label_map)} cluster labels from task codes: {label_map}")

ari = adjusted_rand_index(labels, labels)
nmi = normalized_mutual_info(labels, labels)
purity = purity_score(labels, labels)
sil = silhouette_score(emb_norm, labels)
ch = calinski_harabasz_score(emb_norm, labels)

print(f"\nClustering quality:")
print(f"  ARI={ari:.4f}  NMI={nmi:.4f}  Purity={purity:.4f}")
print(f"  Silhouette={sil:.4f}  Calinski-Harabasz={ch:.2f}")

Derived 1 cluster labels from task codes: {'32': 0}

Clustering quality:
  ARI=1.0000  NMI=1.0000  Purity=1.0000
  Silhouette=0.0000  Calinski-Harabasz=0.00


## 6. Full Quality Report (Single Call)

Instead of calling each metric individually, `evaluate_embedding_quality` runs everything in one shot.

In [13]:

report = evaluate_embedding_quality(
    embeddings=emb_norm,
    labels=labels,
    hubness_k=10,
    model_name="LLM Task Descriptions",
)

print(report)

Embedding Quality Report: LLM Task Descriptions
  Samples=323  Dims=1536
Intrinsic (Cosine Similarity)
  mean=-0.0029  std=0.1770  min=-0.4042  max=0.9809
Clustering
  ARI=1.0000  NMI=1.0000  Purity=1.0000
  Silhouette=0.0000  Calinski-Harabasz=0.00
Geometry & Pathology
  Effective Rank=70.87  Mean Sim-to-Mean=0.0157
  Hubness Skewness=0.2935 (k=10)
